In [0]:
print("Ambiente do MVP de gramados funcionando!")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.mvp_gramados")

spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.mvp_gramados.arquivos
""")

print("Estrutura do projeto criada!")

In [0]:
pasta = "/Volumes/workspace/mvp_gramados/arquivos/"

# Lê os CSVs usando a primeira linha como nomes das colunas.
# multiLine permite que uma célula contenha quebras de linha,
# como acontece nas fontes da pesquisa de gramados.
partidas = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(pasta + "campeonato-brasileiro-full.csv")
)

gramados = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(pasta + "gramados.csv")
)

print("Quantidade de partidas:", partidas.count())
print("Quantidade de registros de gramados:", gramados.count())

display(gramados.limit(5))

In [0]:
# Salva os dados brutos como tabelas Delta
partidas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_gramados.bronze_partidas")

gramados.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_gramados.bronze_gramados")

# Confere as quantidades lendo diretamente das tabelas salvas
display(spark.sql("""
    SELECT 'bronze_partidas' AS tabela, COUNT(*) AS quantidade
    FROM workspace.mvp_gramados.bronze_partidas

    UNION ALL

    SELECT 'bronze_gramados' AS tabela, COUNT(*) AS quantidade
    FROM workspace.mvp_gramados.bronze_gramados
"""))